# SRD Pose Emergency Core 코드리뷰 정리

대상 파일: `srd_pose_emergency_core.py`

이번 노트북은 Pose Core 한 파일만 기준으로 정리했다. nav, stt, 관제노드 설명은 제외하고 **코어가 프레임 1장을 받아 어떤 순서로 판단을 내리는지**에 집중한다.

## 1. 한 줄 요약

이 코어는 **OpenCV BGR frame -> pose 추론 -> observation -> posture -> motion -> trapped -> emergency_level** 순서로 처리한 뒤, `annotated frame`과 `structured result list`를 반환하는 분석 엔진이다.

## 2. 전체 흐름

```text
OpenCV BGR frame
  -> YOLO Pose 추론
  -> 유효 keypoint 필터링
  -> observation 판정
  -> posture 판정
  -> motion 계산 / 분류
  -> trapped 보조 판정
  -> state duration 계산
  -> emergency_level 결정
  -> annotated frame + result list 반환
```

### 🧠 Vision Node & Core Logic Flowchart

이 셀이 일반 마크다운 글로 보인다면 깃허브(web)에서 보시거나, VSCode에서 `Markdown Preview Mermaid Support` 확장을 설치하시면 다이어그램으로 자동 변환되어 보입니다!

```mermaid
flowchart TD
    SUB[Subscribe: /camera/image_raw/compressed]
    PUB1[Publish: /robot6/emergency_level]
    PUB2[Publish: /robot6/image_result/compressed]

    SUB --> DEC[Decode JPG to BGR Frame]
    DEC --> ENGINE[[Call PoseEmergencyEngine.analyze_frame]]

    ENGINE --> YOLO[YOLO11 Pose 추론]
    YOLO --> VIS{Visibility Classification<br>가시성 분류}
    
    VIS -->|얼굴+양어깨+양골반 관측| C1(FULL_BODY<br>Extract: Shoulder & Hip Center)
    VIS -->|얼굴+양어깨만 관측| C2(UPPER_BODY<br>Extract: Shoulder Center)
    VIS -->|신체 일부만 관측됨| C3(PARTIAL<br>Extract: BBox Center)
    VIS -->|아주 멀거나 흐릿함 - 분석 제외| LOW_CONF(LOW_CONF)

    C1 --> POS{Posture Classification}
    C2 --> POS
    C3 --> POS

    POS --> MOT[Motion Analysis over time<br>NONE / LOW / ACTIVE]
    MOT --> DECIDE{Rules & Time Decision<br>ex: LYING + NONE > 7.0s}
    DECIDE --> RES((Result:<br>CRITICAL / WARNING / CAUTION / NORMAL))

    RES --> ENC[Encode Annotated Frame to JPG]
    RES --> PUB1
    ENC --> PUB2
```


## 3. 코드 동작 설명

### 입력
- `frame`: `np.ndarray`
- 형식: OpenCV 기준 BGR image

### 출력
- `annotated`: `np.ndarray`
- `results`: `List[dict]`
- `results_to_json(results)`: `str`
- `extract_frame_emergency_level(results)`: `str | None`

### 중요
Pose Core 내부에는 topic, service, action이 없다. 외부 노드가 Python API를 직접 호출하는 구조다.

## 4. 핵심 함수

| 함수 | 역할 |
|---|---|
| `AnalyzerConfig` | 임계값, 모델 경로, 시각화 옵션을 모아둔 설정 클래스 |
| `_classify_visibility` | FULL_BODY / UPPER_BODY / PARTIAL / LOW_CONF 판정 |
| `_classify_posture` | shoulder_tilt, head_drop_ratio, torso_angle 기반 자세 판정 |
| `_motion_value`, `_classify_motion` | 프레임 간 keypoint 이동량으로 움직임 계산 |
| `_possible_trapped` | 매몰 의심 여부 계산 |
| `_state_duration` | track_id별 관측 시간, 상태 지속 시간 계산 |
| `_decide` | 최종 emergency level 결정 |
| `_pack_result` | structured result dict 생성 |
| `analyze_frame_with_results` | 메인 진입점 |

## 5. 판정 로직 요약

### Observation
유효 keypoint 수와 상/하체 존재 여부로 `LOW_CONF`, `FULL_BODY`, `UPPER_BODY`, `PARTIAL`을 나눈다.

### Posture
- 전신: `shoulder_tilt`, `head_drop_ratio`, `torso_angle`, `aspect`를 함께 사용
- 상반신: `shoulder_tilt`, `head_drop_ratio` 중심으로 사용

### Motion
이전 프레임과 현재 프레임의 keypoint 차이로 움직임을 정량화하고, `motion_window`로 smoothing 한다.

### Emergency Level
`observation + posture + motion + trapped + state_sec` 조합으로 최종 등급을 만든다.

## 6. 결과 dict 스키마

- `track_id`
- `bbox`
- `observation`
- `posture`
- `motion`
- `emergency_level`
- `trapped`
- `seen_sec`, `state_sec`
- `shoulder_tilt`, `head_drop_ratio`, `torso_angle`
- `motion_smooth`, `motion_upper`, `motion_core`
- `rep_point_px`, `rep_point_method` 등 대표점 필드

## 7. 코드 주석 기준

Python에서는 **싱글쿼트(`'`)로 주석을 쓰지 않는다.**

- 설명 주석: `#`
- 함수/클래스 설명: `"""docstring"""`

코드리뷰에서는 이 기준으로 정리하는 것이 맞다.

In [ ]:
# v0.000
# file: comment_guideline_example.py
# date: 2026-03-11
# changes:
# - 코드리뷰용 주석 예시

# 1) observation 판정
# 2) posture 판정
# 3) motion 계산

def analyze_frame_with_results(frame):
    """프레임 1장을 분석해서 annotated와 result list를 반환한다."""
    pass


## 8. 코드리뷰에서 꼭 말할 것

1. Pose Core는 ROS 노드가 아니라 라이브러리다.
2. 입력은 BGR frame 1장, 출력은 annotated frame과 structured result다.
3. observation이 뒤 로직의 큰 분기 기준이다.
4. posture와 motion은 단발 수치가 아니라 임계값 + smoothing + duration을 함께 본다.
5. 최종 emergency level은 `state_sec`까지 포함한 시간 누적형 판정이다.

### 🚀 판단 기준 요약 (Evaluation Criteria Summary)

현재 파이썬 코드(`rescue_vision_core.py`)에 하드코딩된 주요 판별 조건절을 알기 쉽게 정리한 표입니다.

| 단계 (Step) | 분류 (Class) | 판단 기준 (Thresholds) |
|---|---|---|
| **가시성<br>(Visibility)** | `FULL_BODY` | 머리, 양 어깨, 양 골반 관절의 신뢰도가 모두 0.5 이상일 때 |
| | `UPPER_BODY` | 골반은 잘렸지만 머리와 양 어깨까지만 확실히 렌더링될 때 |
| | `PARTIAL` | 머리와 어깨마저 불확실하게 짤리거나 겹쳐서 안 보일 때 |
| **자세<br>(Posture)** | `LYING` | (전신 전용) Bbox 가로가 세로의 2배 이상 길거나 척추(어깨-골반) 각도가 45도 넘게 기움 |
| | `COLLAPSED` | 허리/어깨선이 45도 이상 심하게 무너졌거나, 고개 꺾임(drop_ratio)이 0.5~0.6 이상일 때 |
| | `LEANING` | 어깨가 15도 이상 기울었거나 고개가 30% 이상 꺾여 불안정한 자세를 아슬아슬하게 유지 |
| | `NORMAL` | 위 항목에 해당하지 않는 수직에 가까운 안정적인 서기/앉기 자세 |
| **움직임<br>(Motion)** | `ACTIVE` | 부드러운 움직임(smooth)이나 상체 움직임(upper) 픽셀 평균이 기준치를 초과함 |
| | `LOCAL_ONLY` | 상체/팔은 발버둥 치지만 코어(어깨/골반 중심)가 꼼짝없이 고정된 경우 (깔림 의심) |
| | `LOW` | 작은 떨림 수준만 감지되는 미세한 움직임 |
| | `NONE` | 움직임 제로. 의식을 잃었거나 기절 의심 |
| **최종 판정<br>(Emergency)**| `CRITICAL` 🚨 | (전신) 누움/붕괴 자세 + 움직임 `NONE` 상태가 **7.0초 이상** 변함없이 지속 |
| | `WARNING` 🔴 | (전신) 누움/붕괴 + 미세 움직임 `LOW/NONE` 상태가 **5.5초 이상** 지속 |
| | `CAUTION` 🟡 | 몸이 기울었거나 아무 자세에서나 움직임이 `LOW/NONE`으로 **4.5초 이상** 지속 |
| | `NORMAL` 🟢 | 그 외의 모든 안전/활동 반경 내의 판정들 |


---

### 📡 ROS2 Topics & 🧭 Code Line References

#### 1. ROS2 Topics (`rescue_vision_node.py`)

| Topic Name Parameter | Default Value | Message Type | I/O |
|---|---|---|---|
| `input_image_topic` | `/camera/image_raw/compressed` | `sensor_msgs/CompressedImage` | **Subscribe** |
| `emergency_level_topic` | `/robot6/emergency_level` | `std_msgs/String` | **Publish** |
| `image_result_topic` | `/robot6/image_result/compressed` | `sensor_msgs/CompressedImage` | **Publish** |

#### 2. 주요 로직 라인 넘버 (Line Numbers)

> **`rescue_vision_node.py` (ROS2 Wrapper)**
- `__init__` (파라미터/Pub/Sub 선언): Line 47~103
- `image_callback` (메인 루프): Line 148~180
- 디코딩/인코딩 유틸 (`_decode_compressed_image` 등): Line 108~143

> **`rescue_vision_core.py` (Core Engine)**
- `_classify_visibility` (가시성 판단): Line 319~361
- `_classify_posture` (자세 수학적 계산): Line 366~497
- `_classify_motion` (시간/프레임 비교 움직임 계산): Line 557~574
- `_decide` (최종 위급수준 시간적 판정): Line 607~683
- `analyze_frame_with_results` (YOLO + 위 로직 종합 오케스트레이터): Line 776~910


---
### 💻 핵심 함수 원본 소스코드 (Reference Snippets)
앞서 정리한 라인 넘버 목차에 해당하는 실제 핵심 로직 소스코드입니다.


#### rescue_vision_node.py - __init__ (Line 47~103)


In [ ]:
    def __init__(self):
        # 부모 클래스(Node)의 초기화 함수를 호출해 ROS 네트워크에 이 노드의 이름을 알립니다.
        super().__init__("srd_pose_emergency_node")

        # --------------------------------------------------------------
        # ROS2 파라미터(Parameter) 선언 및 가져오기
        # 파라미터는 노드를 실행할 때(launch 파일이나 터미널 명령어) 동적으로 바꿀 수 있는 설정값들입니다.
        # 코드 수정 없이도 토픽 이름이나 모델 경로를 바꿀 수 있어 유연합니다.
        # --------------------------------------------------------------
        # 1. 사용할 파라미터들의 이름과 '기본값(default)'을 미리 시스템에 등록합니다.
        self.declare_parameter("model_path", "yolo11n-pose.pt")
        self.declare_parameter("input_image_topic", "/camera/image_raw/compressed")
        self.declare_parameter("emergency_level_topic", "/robot6/emergency_level") # 최종 판정 문자열 발행
        self.declare_parameter("image_result_topic", "/robot6/image_result/compressed") # 뼈대 그려진 그림 발행
        self.declare_parameter("publish_annotated", True) # 뼈대 이미지를 실제로 보낼지 말지 On/Off 스위치
        self.declare_parameter("show_debug", True)        # 이미지 밑에 글씨(각도 등 디버깅 정보)를 띄울지 On/Off 스위치

        # 2. 등록된 구멍에서 실제 값들을 꺼내와서 파이썬 변수에 저장합니다.
        model_path = self.get_parameter("model_path").get_parameter_value().string_value
        input_image_topic = self.get_parameter("input_image_topic").get_parameter_value().string_value
        emergency_level_topic = self.get_parameter("emergency_level_topic").get_parameter_value().string_value
        image_result_topic = self.get_parameter("image_result_topic").get_parameter_value().string_value
        
        # boolean(참/거짓) 스위치들은 콜백 함수 안에서 매번 확인해야 하므로 self 변수로 격상시켜 저장합니다.
        self.publish_annotated = self.get_parameter("publish_annotated").get_parameter_value().bool_value
        show_debug = self.get_parameter("show_debug").get_parameter_value().bool_value

        # --------------------------------------------------------------
        # 분석 코어 엔진 설정 및 생성 (다른 파일에 있는 로직 덩어리를 불러옵니다)
        # --------------------------------------------------------------
        # 엔진에 전달할 세팅 꾸러미(Config)를 만듭니다.
        cfg = AnalyzerConfig(model_path=model_path, show_debug=show_debug)
        # 세팅 꾸러미를 넣어서 메인 분석 뇌(Engine) 하나를 생성해둡니다.
        self.engine = PoseEmergencyEngine(cfg)

        # --------------------------------------------------------------
        # ROS2 퍼블리셔(발행자)와 서브스크라이버(구독자) 생성
        # --------------------------------------------------------------
        # 분석 결과 문자열을 바깥(ROS 네트워크)으로 뿌려줄 스피커(Publisher) 생성 (큐 사이즈 10)
        self.emergency_level_pub = self.create_publisher(String, emergency_level_topic, 10)
        
        # 뼈대가 예쁘게 그려진 이미지를 바깥으로 뿌려줄 스피커 생성
        self.image_result_pub = self.create_publisher(CompressedImage, image_result_topic, 10)

        # 원본 카메라 압축 이미지를 받을 귀(Subscriber) 생성
        # 이미지가 한 장 들어올 때마다 `self.image_callback` 이라는 지정된 함수가 자동으로 호출됩니다!
        self.image_sub = self.create_subscription(
            CompressedImage,
            input_image_topic,
            self.image_callback,
            10,
        )

        # 프로그램이 잘 켜졌다고 터미널 창에 초록색 글씨로 안내문(로그)을 하나 남겨줍니다.
        self.get_logger().info(
            f"SRD Pose Emergency Node started. input={input_image_topic}, emergency_level={emergency_level_topic}, annotated={image_result_topic}"
        )


#### rescue_vision_node.py - image_callback (Line 148~180)


In [ ]:
    def image_callback(self, msg: CompressedImage) -> None:
        """ROS2 시스템이 카메라 압축 영상을 받을 때마다 자동으로 호출해 주는 함수입니다.
        이 프로그램의 '심장이 뛰는 곳'입니다.
        """
        try:
            # 1) 통신용 압축 파일(msg)을 AI가 볼 수 있는 그림판(frame)으로 변환합니다.
            frame = self._decode_compressed_image(msg)

            # 2) "분석 코어 엔진아, 이 그림 받아서 뼈대 그리고 위급 상황인지 알려줘!" 하고 지시합니다.
            # 엔진은 뼈대 그려진 사진(annotated)과, 최종 위급 단계 글자(emergency_level)를 줍니다.
            annotated, emergency_level = self.engine.analyze_frame_with_emergency_level(frame)

            # 3) 엔진 결과 배포 (퍼블리시)
            # 만약 화면에 사람이 단 한 명이라도 있어서 위급수준 결과(emergency_level)가 나왔다면:
            if emergency_level is not None:
                # ROS2 문자열 빈 봉투(String)를 만들고
                em_msg = String()
                # 글자를 적어서 넣은 뒤
                em_msg.data = emergency_level
                # 바깥세상(ROS 네트워크)으로 외칩니다! "현재 상태: CRITICAL!!"
                self.emergency_level_pub.publish(em_msg)

            # 4) 시각화 이미지 배포 (퍼블리시)
            # 파라미터에서 뼈대 그림을 만들라고 켜놨다면(publish_annotated == True):
            if self.publish_annotated:
                # 뼈대가 그려진 그림(annotated)을 다시 통신용 압축 파일로 포장합니다.
                annotated_msg = self._encode_compressed_image(annotated, msg.header.stamp)
                # 바깥세상으로 사진 통째로 쏴줍니다. (RQT 같은 모니터링 툴에서 볼 수 있음)
                self.image_result_pub.publish(annotated_msg)

        # 프로그램이 돌다가 뭔가 에러가 나면 멈춰버리지 않고 빨간 글씨(Error)로 에러 내용만 띄웁니다.
        except Exception as exc:
            self.get_logger().error(f"image_callback failed: {exc}")


#### rescue_vision_core.py - _classify_visibility (Line 319~361)


In [ ]:
    def _classify_visibility(
        self,
        keypoints: np.ndarray,
        kp_conf: np.ndarray,
        shape: Tuple[int, int, int],
    ) -> str:
        """사람의 몸이 카메라에 얼마나 노출되었는지 4단계 가시성 상태로 분류합니다.

        [반환값 의미]
        - LOW_CONF   : 신뢰할 수 있는 관절점이 너무 적어 분석 불가 (오탐지이거나 너무 멀리 있음)
        - FULL_BODY  : 상체와 하체가 모두 충분히 확보된 상태 (가장 정확한 전신 분석 가능)
        - UPPER_BODY : 상체는 잘 보이지만 하체가 가려지거나 잘린 상태 (상반신 분석 모드로 동작)
        - PARTIAL    : 상하체 전체적으로 관절이 부족하지만 인식 무시는 아닌 상태 (일부 노출)
        """
        # 먼저 상체와 하체에서 '유효한(믿을만한)' 관절들의 번호표만 모아옵니다.
        upper_valid = self._valid_indices(keypoints, kp_conf, self.UPPER_IDS, shape)
        lower_valid = self._valid_indices(keypoints, kp_conf, self.LOWER_IDS, shape)
        total_count = len(upper_valid) + len(lower_valid)

        # 전체 유효 관절 수가 설정된 최소치(기본 3개)보다 적으면 '보이지 않음'으로 간주
        if total_count < self.cfg.low_conf_min_kps:
            return "LOW_CONF"

        # [전신 FULL_BODY 판단 조건]
        # 1) 상체 관절이 충분히 보임 (upper_body_min_kps 이상)
        # 2) 왼쪽(11) 혹은 오른쪽(12) 골반 중 하나는 반드시 보여야 함 (상하체 연결점)
        # 3) 골반을 제외한 나머지 하체 관절(무릎, 발목)이 일정 개수 이상 추가로 보여야 함
        hips_ok = 11 in lower_valid or 12 in lower_valid
        extra_lower = len([i for i in lower_valid if i not in (11, 12)]) # 골반 제외 하체 관절 개수

        if (
            len(upper_valid) >= self.cfg.upper_body_min_kps       # 상체 조건 만족
            and hips_ok                                           # 골반 뼈 존재 확인
            and extra_lower >= self.cfg.full_body_extra_lower_kps # 추가 하체 관절 확인
        ):
            return "FULL_BODY"

        # 전신 조건은 만족하지 못했지만, 상체 관절 개수만 충분하다면 상반신으로 판정
        if len(upper_valid) >= self.cfg.upper_body_min_kps:
            return "UPPER_BODY"

        # 상체도 하체도 기준에 못 미치지만, 완전 LOW_CONF는 아닌 애매한 일부 노출 상태
        return "PARTIAL"


#### rescue_vision_core.py - _classify_posture (Line 366~497)


In [ ]:
    def _classify_posture(
        self,
        keypoints: np.ndarray,
        kp_conf: np.ndarray,
        box: np.ndarray,
        visibility: str,
        shape: Tuple[int, int, int],
    ) -> Tuple[str, float, float, float]:
        """사람의 관절 기울기를 삼각함수로 계산하여 현재 자세를 분류합니다.

        [분류 단계]
        - NORMAL(정상) : 특별히 기울어지거나 꺾이지 않은 일반적인 서거나 앉은 자세
        - LEANING(기울어짐) : 어깨나 고개가 살짝 꺾이거나 숙여진 불안정한 자세
        - COLLAPSED(쓰러짐/붕괴) : 뼈대가 극심하게 꺾여 심각한 부상이나 기절이 의심되는 자세
        - LYING(누움) : 몸통 전체가 바닥과 수평에 가깝게 누워있는 자세 (전신 노출일 때만 판정)
        - UNKNOWN(알 수 없음) : 가시성이 확보되지 않아 자세를 파악할 수 없는 상태

        [반환값 튜플]
        - (자세 텍스트, 어깨 기울기 각도, 머리 꺾임 비율, 허리 숙임 각도)
        """
        # 바운딩 박스를 통해 사람 박스의 넓이(bw), 높이(bh), 그리고 가로/세로 비율(aspect)을 잽니다.
        x1, y1, x2, y2 = box.astype(float)
        bw = max(x2 - x1, 1.0) # 박스 넓이 (0으로 나누는 에러를 막기 위해 최소 1.0 설정)
        bh = max(y2 - y1, 1.0) # 박스 높이
        aspect = bw / bh       # 비율이 클수록(가로로 넓을수록) 사람이 누워있을 확률이 높습니다.

        # 얼굴 / 상체 / 하체 주요 관절점(Keypoint)들을 전부 불러와 준비합니다.
        nose = self._get_point(keypoints, kp_conf, 0, shape)
        leye = self._get_point(keypoints, kp_conf, 1, shape)
        reye = self._get_point(keypoints, kp_conf, 2, shape)
        lear = self._get_point(keypoints, kp_conf, 3, shape)
        rear = self._get_point(keypoints, kp_conf, 4, shape)
        ls = self._get_point(keypoints, kp_conf, 5, shape)
        rs = self._get_point(keypoints, kp_conf, 6, shape)

        # 골반(Hip) 점은 '전신'이 다 보일 때만 사용합니다. 상반신 모드일 때는 억지로 추정하지 않습니다.
        lh = self._get_point(keypoints, kp_conf, 11, shape) if visibility == "FULL_BODY" else None
        rh = self._get_point(keypoints, kp_conf, 12, shape) if visibility == "FULL_BODY" else None

        # 양쪽 어깨, 양쪽 골반, 얼굴 이목구비 5점들의 '중간점(무게중심)'을 각각 구합니다.
        shoulder_center = self._safe_mean([ls, rs])
        hip_center = self._safe_mean([lh, rh])
        face_anchor = self._safe_mean([nose, leye, reye, lear, rear])

        # 좌우 어깨 점을 이어 '어깨선이 기울어진 각도'를 계산합니다. (0~90도)
        shoulder_tilt = self._angle_deg(ls, rs) if ls is not None and rs is not None else 0.0

        # 어깨선 가로 길이(픽셀 단위 너비) 측정
        shoulder_span = 0.0
        if ls is not None and rs is not None:
            shoulder_span = abs(float(rs[0] - ls[0]))

        # [측면(옆보기) 방어 로직] 
        # 사람이 몸을 카메라 옆으로 심하게 돌려 서면 2D 상에서는 양 어깨가 겹쳐서 어깨 너비가 확 줄어듭니다.
        # 이 상태에서 고개만 살짝 까딱해도 3D->2D 투영 왜곡 때문에 각도가 엄청 심하게 꺾인 것처럼 오계산됩니다.
        # 따라서 어깨 너비가 사람 박스 전체 가로폭 대비 너무 좁다면(옆모습이라면) 어깨 기울기 값을 아예 무시(0.0)해버립니다.
        if shoulder_span < bw * self.cfg.upper_body_min_shoulder_span_ratio:
            shoulder_tilt = 0.0

        # [고개 꺾임 비율 (head_drop_ratio) 계산 로직]
        # 얼굴의 무게중심(face_anchor)이 어깨 중심선(shoulder_center)에 얼마나 가깝게 밀착되었는지를 봅니다.
        # 목이 꼿꼿하면 거리가 멀리 떨어져 있고(0.0에 가까움), 기절해서 목이 꺾이거나 책상에 엎드리면 거리가 좁아집니다(1.0에 가까움).
        head_drop_ratio = 0.0
        if face_anchor is not None and shoulder_center is not None and shoulder_span > 1.0:
            # 어깨 Y좌표에서 얼굴 Y좌표를 빼서 수직 거리 차이를 잰 후 (음수 방어용 max 0.0)
            head_gap = max(float(shoulder_center[1] - face_anchor[1]), 0.0)
            # 수직 거리를 어깨뼈 너비(개인 신체 비율 기준)로 나눠서 체형에 상관없이 0~1사이 정규화된 꺾임 비율 도출
            head_drop_ratio = 1.0 - min(head_gap / shoulder_span, 1.0)

        torso_angle = 0.0
        if shoulder_center is not None and hip_center is not None:
            # 척추(어깨-골반을 잇는 선)가 곧게 선 수직 기준선(90도)에서 얼마나 벗어났는지 각도 편차를 구합니다.
            torso_angle = abs(90.0 - self._angle_deg(shoulder_center, hip_center))

        # ------------------------------
        # [전신 FULL_BODY 일 때의 자세 판정 규칙]
        # ------------------------------
        if visibility == "FULL_BODY":
            # [LYING - 누움]
            # 1) 전체 바운딩 박스가 사람 키보다 가로로 2배 이상 길쭉해진 경우 (aspect >= 2.00)
            # 2) 또는 척추가 거의 바닥과 수평(torso_angle >= 72도)인 경우 완벽히 누웠다고 봅니다.
            if aspect >= self.cfg.lying_aspect_ratio or torso_angle >= self.cfg.lying_torso_angle_deg:
                return "LYING", shoulder_tilt, head_drop_ratio, torso_angle

            # [COLLAPSED - 쓰러짐/붕괴]
            # 위 누움 단계까진 아니지만, 다음 3개 중 하나라도 심각하게 꺾인 경우 의식을 잃고 쓰러졌다고 간주합니다.
            # 1) 허리가 60도 이상 꺾임 / 2) 어깨가 45도 이상 꺾임 / 3) 고개가 어깨팍에 거의 파묻힘(78% 이상)
            if (
                torso_angle >= self.cfg.collapsed_torso_angle_deg
                or shoulder_tilt >= self.cfg.collapsed_shoulder_tilt_deg
                or head_drop_ratio >= self.cfg.collapsed_head_drop_ratio
            ):
                return "COLLAPSED", shoulder_tilt, head_drop_ratio, torso_angle

            # [LEANING - 기울어짐]
            # 위 붕괴 단계까진 아니지만, 정상적으로 서있지 못하고 자세가 무너지는 중(Leaning)으로 봅니다.
            # 1) 어깨선이 25도 이상 기울어짐 / 2) 고개가 어깨 쪽으로 눈에 띄게(55% 이상) 내려옴
            if (
                shoulder_tilt >= self.cfg.leaning_shoulder_tilt_deg
                or head_drop_ratio >= self.cfg.leaning_head_drop_ratio
            ):
                return "LEANING", shoulder_tilt, head_drop_ratio, torso_angle

            # 위 모든 위급 조건을 전부 피해갔다면 (꼿꼿이 서있거나, 바르게 앉은 다소곳한 모양새)
            return "NORMAL", shoulder_tilt, head_drop_ratio, torso_angle

        # ------------------------------
        # [상반신 UPPER_BODY 일 때의 자세 판정 규칙]
        # ------------------------------
        if visibility == "UPPER_BODY":
            # 상반신만 보일 때는 골반(Hip) 위치를 모르므로 허리 각도(torso_angle)는 쓰지 못합니다.
            # 어깨(shoulder_tilt)와 고개(head_drop_ratio) 단 두 가지의 상태만으로 자세를 추론합니다.

            # [COLLAPSED - 붕괴] 어깨나 고개 중 하나라도 기준치(45도, 78%) 이상 심하게 꺾이면 의심
            if (
                shoulder_tilt >= self.cfg.collapsed_shoulder_tilt_deg
                or head_drop_ratio >= self.cfg.collapsed_head_drop_ratio
            ):
                return "COLLAPSED", shoulder_tilt, head_drop_ratio, torso_angle

            # [LEANING - 기울어짐] 어깨나 고개가 기준치(25도, 55%) 이상 불안정하게 기운 경우
            if (
                shoulder_tilt >= self.cfg.leaning_shoulder_tilt_deg
                or head_drop_ratio >= self.cfg.leaning_head_drop_ratio
            ):
                return "LEANING", shoulder_tilt, head_drop_ratio, torso_angle

            # 안전
            return "NORMAL", shoulder_tilt, head_drop_ratio, torso_angle

        # 전신도 상반신도 아니면(PARTIAL 등) 관절 개수가 모자라므로, 자세를 '모름(UNKNOWN)'으로 예외처리
        return "UNKNOWN", shoulder_tilt, head_drop_ratio, torso_angle


#### rescue_vision_core.py - _classify_motion (Line 557~574)


In [ ]:
    def _classify_motion(self, smooth: float, upper: float, core: float) -> str:
        """계산해낸 수학적 이동량 수치를 바탕으로 움직임(Motion) 크기를 4단계 텍스트로 라벨링합니다."""
        
        # [LOCAL_ONLY - 부분 움직임] : 팔이나 무릎(상체)은 크게 허우적대는데, 몸통 코어(어깨/골반)가 딱 박혀서 안 움직일 때
        # => 어딘가에 깔렸거나 모서리에 끼여서 빠져나오지 못하고 발버둥 치는 안타까운 상황의 주요 힌트가 됩니다.
        if upper >= self.cfg.motion_local_only_upper and core <= self.cfg.motion_local_only_core:
            return "LOCAL_ONLY"
            
        # [ACTIVE - 활발] 전체 움직임 버퍼 평균이나 순간 상체 움직임이 기준을 넘는 원활한 움직임
        if smooth >= self.cfg.motion_active_smooth or upper >= self.cfg.motion_active_upper:
            return "ACTIVE"
            
        # [LOW - 미세 움직임] 활발 수치엔 못 미치지만, 숨을 쉬거나 몸을 떠는 등 작은 수치라도 센싱될 때 (사망 판정 지연용)
        if smooth >= self.cfg.motion_low:
            return "LOW"
            
        # [NONE - 기절/사망] 로봇 카메라 노이즈 수준 이하의 어떤 유의미한 움직임도 없는 서늘한 상태
        return "NONE"


#### rescue_vision_core.py - _decide (Line 607~683)


In [ ]:
    def _decide(
        self,
        visibility: str,
        posture: str,
        motion: str,
        trapped: bool,
        seen_sec: float,
        state_sec: float,
    ) -> str:
        """관측 상태, 자세, 움직임, 그리고 '해당 상태가 얼마나 오랫동안 지속되었는가(state_sec)'를 
        종합하여 최종적으로 화면에 띄울 위급 단계(NORMAL~CRITICAL)를 결정합니다.

        [시간 지연(Time Delay) 로직의 이유]
        사람이 잠깐 신발끈을 묶으려고 허리를 숙이거나 실수로 넘어졌다가 바로 일어나는 경우에도 
        즉시 알람이 울리면 양치기 소년 시스템이 됩니다.
        따라서 특정 위험 자세(예: 누움, 움직임 없음)가 설정된 시간(예: 4.5초, 7초) 이상 
        **꾸준히 유지(지속)**될 때만 위급 단계를 서서히 올립니다.
        """
        # [ANALYZING] 카메라에 사람이 처음 잡힌 직후(기본 1.5초 이내)에는 
        # 움직임 버퍼에 데이터가 덜 쌓여서 오판할 확률이 높으므로 섣불리 판단하지 않고 대기합니다.
        if seen_sec < self.cfg.analyzing_sec:
            return "ANALYZING"

        # ==========================================
        # 조건 분기 1: 전신이 다 보일 때 (가장 확실한 판단 가능)
        # ==========================================
        if visibility == "FULL_BODY":
            # [CRITICAL] 누웠거나 붕괴된 최악의 자세 + 미동조차 없음 + 이게 7초(critical_sec) 이상 지속됨
            # => 심장마비, 완전 기절 등 즉시 출동해야 하는 초위급 상황
            if posture in ("LYING", "COLLAPSED") and motion == "NONE" and state_sec >= self.cfg.critical_sec:
                return "CRITICAL"
                
            # [WARNING] 쓰러졌는데 아주 미세하게 떨고 있거나, 아직 5.5초(warning_sec)밖에 안 지난 경우
            if posture in ("LYING", "COLLAPSED") and motion in ("LOW", "NONE") and state_sec >= self.cfg.warning_sec:
                return "WARNING"
                
            # [CAUTION] 다음 3가지 중 하나에 해당하면 주의(노란불)를 줍니다.
            # 1) 쓰러지진 않았지만 몸이 불안정하게 기운(Leaning) 상태로 4.5초(caution_sec) 이상 지속될 때
            # 2) 자세는 정상(Normal)인데, 5.5초 이상 꼼짝도 안 하고 서있거나 앉아있을 때 (졸도 직전 의심)
            # 3) 움직임이 활발하지 못하고 미세한 상태(LOW)가 4.5초 이상 지속될 때
            if (
                (posture == "LEANING" and motion in ("LOW", "NONE") and state_sec >= self.cfg.caution_sec)
                or (posture == "NORMAL" and motion == "NONE" and state_sec >= self.cfg.warning_sec)
                or (motion == "LOW" and state_sec >= self.cfg.caution_sec)
            ):
                return "CAUTION"
                
            # 위 모든 위험 조건을 피했다면 안전한 상태
            return "NORMAL"

        # ==========================================
        # 조건 분기 2: 상반신만 보일 때 (책상 앞, 벽 뒤)
        # ==========================================
        if visibility == "UPPER_BODY":
            # 상반신만 보이는데 엎드렸거나 고개가 완전히 꺾였고(Collapsed) 움직임도 없다면 경고
            if posture == "COLLAPSED" and motion == "NONE" and state_sec >= self.cfg.warning_sec:
                return "WARNING"
            # 엎드리진 않았지만 활발한 움직임이 4.5초 이상 아예 없다면 주의 (졸거나 쓰러진 상태 의심)
            if motion in ("LOW", "NONE") and state_sec >= self.cfg.caution_sec:
                return "CAUTION"
            return "NORMAL"

        # ==========================================
        # 조건 분기 3: 극히 일부만 보일 때 (잔해물 밑에 깔린 경우 등)
        # ==========================================
        if visibility == "PARTIAL":
            # 팔다리만 허우적대거나 꼼짝 안 하면서 '깔림(Trapped)'이 의심되는 상태로 오래 지속되면
            # 제대로 안 보임에도 불구하고 과감하게 구조 경고를 띄웁니다.
            if trapped and motion == "NONE" and state_sec >= self.cfg.critical_sec:
                return "WARNING"
            if motion in ("LOW", "NONE") and state_sec >= self.cfg.caution_sec:
                return "CAUTION"
            return "NORMAL"

        # 코드가 여기까지 도달할 일은 거의 없지만, 알 수 없는 예외 상황이라면 
        # 안전을 위해 무시하지 않고 일단 CAUTION을 띄워 로봇 관리자가 한 번 쳐다보게 만듭니다. (Fail-safe)
        return "CAUTION"
